# SparseWalker vs SASRec: A100 serving-speed benchmark

This is a **batch=1 online-serving microbenchmark**. It separates SASRec full-prefix recomputation, exact SASRec KV-cached incremental inference, SparseWalker Triton local update, a sparse-temporal Top-16 GPU compute proxy, and final catalog retrieval.

Important: the temporal SWG number measures scoring/read over ~96 graph-visited candidates per head (matching the measured HNSW search budget). It **does not yet include actual GPU graph pointer-chasing**, so treat it as a lower-bound/proxy rather than production latency.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'
BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src')
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


## Run

Watch these lines:
- `SASREC_FULL_PREFIX`
- `SASREC_INCREMENTAL_KV`
- `WALKER_TRITON_LOCAL_UPDATE_US`
- `WALKER_TEMPORAL_SPARSE_2L_PROXY_US`
- `WALKER_ENCODER_PROXY_US`
- `SASREC_DENSE_CATALOG`
- `WALKER_TERMINAL_512_CANDIDATES_US`


In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/benchmarks/run_serving_speed.py'
sys.argv=[SCRIPT]
print('INPROCESS SERVING SPEED BENCH START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('INPROCESS SERVING SPEED BENCH END',flush=True)


## Inspect saved result

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_speed/serving_speed.json')
print(json.dumps(json.loads(p.read_text()),indent=2) if p.exists() else 'result not found')
